In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

In [2]:
from google.colab import files

uploaded = files.upload()
df = pd.read_csv(next(iter(uploaded)))

Saving merged_policies.csv to merged_policies.csv


In [3]:
df = df.drop(columns=["Launch_Date"], errors="ignore")

text_columns = [
    "Policy_Name",
    "Policy_Type",
    "Department",
    "Target_Group",
    "Benefits_provided",
    "Policy_Description"
]

In [4]:
for col in text_columns:
    df[col] = df[col].fillna("").astype(str)

In [5]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)
    text = text.replace("|", ". ")
    text = text.replace("•", ". ")
    text = text.replace("·", ". ")
    text = re.sub(r'[\u2018\u2019]', "'", text)
    text = re.sub(r'[\u201C\u201D]', '"', text)
    text = re.sub(r'[\u2013\u2014]', "-", text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    text = re.sub(r'([,.;:!?])(?=[A-Za-z₹])', r'\1 ', text)
    text = re.sub(r'₹\s+', '₹', text)
    text = re.sub(r'\s+%', '%', text)
    text = re.sub(r'(\d)\s+-\s+(\d)', r'\1-\2', text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'\s*,\s*,+', ', ', text)
    text = re.sub(r'\s*:\s*', ': ', text)
    text = re.sub(r'\s*;\s*', '; ', text)
    return text.strip(" .,:;")

In [6]:
for col in text_columns:
    df[col] = df[col].apply(clean_text)


In [7]:
df["policy_text"] = (
    "Policy name: " + df["Policy_Name"] + ". "
    + "Policy type: " + df["Policy_Type"] + ". "
    + "Department: " + df["Department"] + ". "
    + "Target group: " + df["Target_Group"] + ". "
    + "Benefits provided: " + df["Benefits_provided"] + ". "
    + "Policy description: " + df["Policy_Description"] + "."
)

df["policy_text"] = df["policy_text"].apply(clean_text)
df["policy_text"] = df["policy_text"].str.replace(r'\.\s*\.', '.', regex=True)
df["policy_text"] = df["policy_text"].str.replace(r'\s+', ' ', regex=True).str.strip()

In [8]:
empty_policies = df["policy_text"].eq("").sum()
duplicate_policies = df["policy_text"].duplicated().sum()

In [9]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Empty policies:", empty_policies)
print("Duplicate policies:", duplicate_policies)
print("Missing values:")
print(df[text_columns].eq("").sum())

Rows: 4745
Columns: 7
Empty policies: 0
Duplicate policies: 0
Missing values:
Policy_Name            0
Policy_Type            0
Department            79
Target_Group          82
Benefits_provided      0
Policy_Description     0
dtype: int64


In [10]:
df.to_csv("cleaned_policies.csv", index=False)

files.download("cleaned_policies.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>